In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("ExpenseAnomalyDetection") \
    .master("local[*]") \
    .getOrCreate()

In [2]:
df = spark.read.csv("transactions.csv", header=True, inferSchema=True)

print("Raw transactions:")
df.show()

Raw transactions:
+-------+----------------+-------------------+-------------+-------+
|user_id|transaction_date|              month|     category| amount|
+-------+----------------+-------------------+-------------+-------+
|      1|      2026-01-03|2026-01-01 00:00:00|    Groceries| 1450.0|
|      1|      2026-01-05|2026-01-01 00:00:00|         Rent|12000.0|
|      1|      2026-01-08|2026-01-01 00:00:00|    Utilities|  850.5|
|      1|      2026-01-10|2026-01-01 00:00:00|    Transport|  300.0|
|      1|      2026-01-15|2026-01-01 00:00:00|    Groceries|  980.0|
|      1|      2026-01-25|2026-01-01 00:00:00|     Shopping| 9800.0|
|      2|      2026-01-04|2026-01-01 00:00:00|         Rent|11000.0|
|      2|      2026-01-06|2026-01-01 00:00:00|    Groceries| 1200.0|
|      2|      2026-01-12|2026-01-01 00:00:00|Entertainment|  600.0|
|      2|      2026-01-14|2026-01-01 00:00:00|    Utilities|  720.0|
|      2|      2026-01-20|2026-01-01 00:00:00|   Healthcare|  450.0|
|      3|      2

In [3]:
user_monthly = df.groupBy("user_id", "month") \
    .agg(F.sum("amount").alias("total_spent")) \
    .orderBy("user_id", "month")

print("Monthly spend per user:")
user_monthly.show()

Monthly spend per user:
+-------+-------------------+-----------+
|user_id|              month|total_spent|
+-------+-------------------+-----------+
|      1|2026-01-01 00:00:00|    25380.5|
|      2|2026-01-01 00:00:00|    13970.0|
|      3|2026-01-01 00:00:00|    13600.0|
+-------+-------------------+-----------+



In [4]:
user_stats = df.groupBy("user_id") \
    .agg(F.avg("amount").alias("avg_amount"), F.stddev("amount").alias("std_amount"))

In [5]:
user_stats.toPandas().to_csv("user_stats.csv", index=False)

In [6]:
joined = df.join(user_stats, on="user_id")

In [7]:
anomalies = joined.filter(
    F.col("amount") > (F.col("avg_amount") + 1.5 * F.col("std_amount"))
)

print("Potentially unusual / large one-time expenses:")
anomalies.select("user_id", "transaction_date", "category", "amount").show()

Potentially unusual / large one-time expenses:
+-------+----------------+--------+-------+
|user_id|transaction_date|category| amount|
+-------+----------------+--------+-------+
|      2|      2026-01-04|    Rent|11000.0|
+-------+----------------+--------+-------+



In [8]:
user_monthly.toPandas().to_csv("monthly_totals.csv", index=False)

In [9]:
anomalies.toPandas().to_csv("anomalies.csv", index=False)

In [10]:
spark.stop()